First, create a new conda environment named BI2025 and install the required packages from requirements.txt


In [ ]:
#!conda create -n BI2025 python=3.11 -y
#!conda activate BI2025
#!pip install -r requirements.txt

In [44]:
# DO NOT MODIFY OR COPY THIS CELL!!
# Note: The only imports allowed are Python's standard library, pandas, numpy, scipy, matplotlib, seaborn and scikit-learn
import numpy as np
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import datetime
import typing
import requests
import time
import shutil
import json
import arff
import shap
from starvers.starvers import TripleStoreEngine

from sklearn.base import TransformerMixin, ClassifierMixin
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)

## Graph-based documentation preliminaries

**!!!IMPORTANT!!!**

Everytime you work on this notebook, enter your student ID in the `executed_by` variable so that the cell executions are accredited to you.

In [2]:
executed_by ='stud-id_01227213'  # Replace the digits after "id_" with your own student ID

Set your group and student IDs. Do this only once.

In [3]:
# group id for this project
group_id = "088"  # Replace the digits with your group id

ID_STUDENT_A = "12537811"  # Replace with student A's student ID
ID_STUDENT_B = "12233000"  # Replace with student B's student ID

# Students working on this notebook
student_a = f"stud-id_{ID_STUDENT_A}"  # Replace the digits after "id_" with student A's student ID
student_b = f"stud-id_{ID_STUDENT_B}"  # Replace the digits after "id_" with student B's student ID

In [4]:
# Roles. Don't change these values.
code_writer_role = "code_writer"
code_executor_role = "code_executor"

Setup the starvers API for logging your steps into our server-sided graph database.

In [5]:
get_endpoint = "https://starvers.ec.tuwien.ac.at/BI2025"
post_endpoint = "https://starvers.ec.tuwien.ac.at/BI2025/statements"
engine = TripleStoreEngine(get_endpoint, post_endpoint, skip_connection_test=True)

Use these prefixes in your notebooks. You can extend this dict with your prefixes of additional ontologies that you use in this notebook. Replace 00 with your group id

In [6]:
prefixes = {
    "xsd": "http://www.w3.org/2001/XMLSchema#",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
    "foaf": "http://xmlns.com/foaf/0.1/",
    "prov": "http://www.w3.org/ns/prov#",
    "sc": "https://schema.org/",
    "cr": "http://mlcommons.org/croissant/",
    "mls": "http://www.w3.org/ns/mls#",
    "mlso": "http://w3id.org/mlso",
    "siu": "https://si-digital-framework.org/SI/units/",
    "siq": "https://si-digital-framework.org/SI/quantities/",
    "qudt": "http://qudt.org/schema/qudt/",
    "unit": "http://qudt.org/vocab/unit/",
    "": f"https://starvers.ec.tuwien.ac.at/BI2025/{group_id}/",
}

prefix_header = "\n".join([f"PREFIX {k}: <{v}>" for k, v in prefixes.items()]) + "\n\n"

Ontologies to use
* Provenance of the experiment process
    * PROV-O: 
        * doc: https://www.w3.org/TR/prov-o/
        * serialization: https://www.w3.org/ns/prov-o
* Data used and created
    * schema.org - Dataset: 
        * doc: https://schema.org/Dataset
        * serialization: https://schema.org/version/latest/schemaorg-current-https.ttl
    * Crossaint
        * doc: https://docs.mlcommons.org/croissant/docs/croissant-spec.html
        * serialization: https://github.com/mlcommons/croissant/blob/main/docs/croissant.ttl
* ML experiments performed
    * MLSO: 
        * doc: https://github.com/dtai-kg/MLSO
        * doc: https://dtai-kg.github.io/MLSO/#http://w3id.org/
        * serialization: https://dtai-kg.github.io/MLSO/ontology.ttl
* Measurements, Metrics, Units
    * QUDT
        * doc:https://qudt.org/
        * doc: https://github.com/qudt/qudt-public-repo
        * serialization: https://github.com/qudt/qudt-public-repo/blob/main/src/main/rdf/schema/SCHEMA_QUDT.ttl
    * SI Digital Framework
        * doc: https://github.com/TheBIPM/SI_Digital_Framework/blob/main/SI_Reference_Point/docs/README.md
        * doc: https://si-digital-framework.org/
        * doc: https://si-digital-framework.org/SI
        * serialization: https://github.com/TheBIPM/SI_Digital_Framework/blob/main/SI_Reference_Point/TTL/si.ttl
    * Quantities and Units
        * doc: https://www.omg.org/spec/Commons
        * serialization: https://www.omg.org/spec/Commons/QuantitiesAndUnits.ttl

Use this function to record execution times.

In [7]:
def now() -> str:
    """
    Returns the current time in ISO 8601 format with UTC timezone in the following format:
    YYYY-MM-DDTHH:MM:SS.sssZ
    """
    timestamp = datetime.datetime.now(datetime.timezone.utc)
    timestamp_formated = timestamp.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

    return timestamp_formated

Register yourself in the Knowledge Graph using ProvO. Change the given name, family name and immatriculation number to reflect your own data.

In [8]:
# Ontologies used: foaf, prov, IAO
reigstration_triples_a = [
    f":{student_a} rdf:type foaf:Person .",
    f":{student_a} rdf:type prov:Agent .",
    f':{student_a} foaf:givenName "Maria" .',
    f':{student_a} foaf:familyName "Salmasian" .',
    f":{student_a} <http://vivoweb.org/ontology/core#identifier> :{student_a} .",
    f":{student_a} rdf:type <http://purl.obolibrary.org/obo/IAO_0000578> .",
    f':{student_a} <http://www.w3.org/2000/01/rdf-schema#label> "Immatriculation number" .',
    f':{student_a} <http://purl.obolibrary.org/obo/IAO_0000219> "{ID_STUDENT_A}"^^xsd:string .',
]

reigstration_triples_b = [
    f":{student_b} rdf:type foaf:Person .",
    f":{student_b} rdf:type prov:Agent .",
    f':{student_b} foaf:givenName "Brais" .',
    f':{student_b} foaf:familyName "Guardado Collazo" .',
    f":{student_b} <http://vivoweb.org/ontology/core#identifier> :{student_b} .",
    f":{student_b} rdf:type <http://purl.obolibrary.org/obo/IAO_0000578> .",
    f':{student_b} <http://www.w3.org/2000/01/rdf-schema#label> "Immatriculation number" .',
    f':{student_b} <http://purl.obolibrary.org/obo/IAO_0000219> "{ID_STUDENT_B}"^^xsd:string .',
]

role_triples = [
    f":{code_writer_role} rdf:type prov:Role .",
    f":{code_executor_role} rdf:type prov:Role .",
]


engine.insert(reigstration_triples_a, prefixes=prefixes)
engine.insert(reigstration_triples_b, prefixes=prefixes)
engine.insert(role_triples, prefixes=prefixes)

**What not do do**

Do not use [blank nodes](https://www.w3.org/wiki/BlankNodes).

PROV-O uses blank nodes to connect multiple elements with each other.
Such blank nodes (such as _:association) should not be used.
Instead, assign a fixed node ID such as
:5119fcd7-b571-41e0-9464-a37c7be0f574 by generating them outside of the
notebook.
We suggest that, for each setting where such a blank node is needed to
connect multiple elements, you create a unique hash (using uuid.uuid4())
and keep this as hard-coded identifier for the blank node. The template
notebook contains examples of this. Do *not* use these provided values,
as otherwise, your provenance documentations will all be connected via
these identifiers!
Also, do not generate them dynamically in every cell execution, e.g. by
using uuid.uuid4() in a cell. This would generate many new linking nodes
for connecting the same elements.
Compute one for each node (cell) where you need them and make sure to
use the same one on each re-execution of the notebook.

## Path

In [9]:
dry_bean_data_path = os.path.join("data", "Dry_Bean_Dataset.arff")

## Business Understanding 

In [10]:
## Each Activity that follows is part of the Business Understanding Phase

business_understanding_phase_executor = [
    ":business_understanding_phase rdf:type prov:Activity .",
    ':business_understanding_phase rdfs:label "Business Understanding Phase" .',  ## Phase 1: Business Understanding
]
engine.insert(business_understanding_phase_executor, prefixes=prefixes)


In [11]:
#############################################
# Documentation
#############################################

data_src_and_scenario_comment = """
The data source is the Dry Bean Dataset, manually collected by the department of computer engineering of the Selcuk University, Turkey. It contains 13611 instances of 7 different dry bean varieties processed into 16 different morphological features (e.g. area, perimeter, major axis length...) via computer vision.

According to the literature, dry beans are the most important and most produced pulses over the world. Assessment of the best seed is the main problem for dry bean producers and markets. Traditionally, bean classification is performed manually, which is a difficult process as well as very time consuming and inefficient at scale. Misclassification errors can also occur, which lead to quality inconsistencies and customer rejections.
"""

business_objectives_comment = """
Two main business objectives can be identified: to automate the dry bean classification problem, cutting operational costs and to minimize misclassification errors induced by this sorting process.
"""

business_success_criteria_comment = """
Some business success criteria tied to the objectives:
- Reduce customer returns by 5%.
- Cut labor costs by 30%.
- 50% increase in sorting speed.
"""

data_mining_goals_comment = """
We propose the following goals:
- Training of two multiclass classification models.
- If model supports it, quantify feature importance and explainability.
"""

data_mining_success_criteria_comment = """
- Trained classifier achieves >95% accuracy.
- Identification of the 5 most important features for classification.
"""

ai_risk_aspects_comment = """
We deem the task of dry bean classification relatively harmless in terms of possible AI Risks identified by the EU AI Act. However, some considerations must be noted:

- Data bias and generalization: the dataset originates from a controlled set of images. Trained models may rely on specific lighting conditions or zoom/scale, failing when faced with real world production environments
- ML Security: computer vision backed models are susceptible to be targeted by adversarial attacks. In this context, a successful attack (intentional or not) could trigger a misclassification, contaminating the batch.
"""


bu_ass_uuid_executor = "b0511130-e18c-49b5-ad82-0c2fac98d7e4"  # Generate once
business_understanding_executor = [
    ":business_understanding rdf:type prov:Activity .",
    ":business_understanding sc:isPartOf :business_understanding_phase .",  # Connect Activity to Parent Business Understanding Phase Activity
    f":business_understanding prov:qualifiedAssociation :{bu_ass_uuid_executor} .",
    f":{bu_ass_uuid_executor} prov:agent :{executed_by} .",
    f":{bu_ass_uuid_executor} rdf:type prov:Association .",
    f":{bu_ass_uuid_executor} prov:hadRole :{code_executor_role} .",
]
engine.insert(business_understanding_executor, prefixes=prefixes)


business_understanding_data_executor = [
    # 1a
    ":bu_data_source_and_scenario rdf:type prov:Entity .",
    ":bu_data_source_and_scenario prov:wasGeneratedBy :business_understanding .",
    ':bu_data_source_and_scenario rdfs:label "1a Data Source and Scenario" .',
    f':bu_data_source_and_scenario rdfs:comment """{data_src_and_scenario_comment}""" .',
    # 1b
    ":bu_business_objectives rdf:type prov:Entity .",
    ":bu_business_objectives prov:wasGeneratedBy :business_understanding .",
    ':bu_business_objectives rdfs:label "1b Business Objectives" .',
    f':bu_business_objectives rdfs:comment """{business_objectives_comment}""" .',
    # 1c
    ":bu_business_success_criteria rdf:type prov:Entity .",
    ":bu_business_success_criteria prov:wasGeneratedBy :business_understanding .",
    ':bu_business_success_criteria rdfs:label "1c Business Success Criteria" .',
    f':bu_business_success_criteria rdfs:comment """{business_success_criteria_comment}""" .',
    # 1d
    ":bu_data_mining_goals rdf:type prov:Entity .",
    ":bu_data_mining_goals prov:wasGeneratedBy :business_understanding .",
    ':bu_data_mining_goals rdfs:label "1d Data Mining Goals" .',
    f':bu_data_mining_goals rdfs:comment """{data_mining_goals_comment}""" .',
    # 1e
    ":bu_data_mining_success_criteria rdf:type prov:Entity .",
    ":bu_data_mining_success_criteria prov:wasGeneratedBy :business_understanding .",
    ':bu_data_mining_success_criteria rdfs:label "1e Data Mining Success Criteria" .',
    f':bu_data_mining_success_criteria rdfs:comment """{data_mining_success_criteria_comment}""" .',
    # 1f
    ":bu_ai_risk_aspects rdf:type prov:Entity .",
    ":bu_ai_risk_aspects prov:wasGeneratedBy :business_understanding .",
    ':bu_ai_risk_aspects rdfs:label "1f AI risk aspects" .',
    f':bu_ai_risk_aspects rdfs:comment """{ai_risk_aspects_comment}""" .',
]
engine.insert(business_understanding_data_executor, prefixes=prefixes)

## Data Understanding

The following pseudo-code & pseudo-documentation may be used as a hint.

In [12]:
## Each Activity that follows is part of the Data Understanding Phase

business_understanding_phase_executor = [
    ":data_understanding_phase rdf:type prov:Activity .",
    ':data_understanding_phase rdfs:label "Data Understanding Phase" .',
]
engine.insert(business_understanding_phase_executor, prefixes=prefixes)


In [13]:
def load_dry_bean_data() -> pd.DataFrame:
    """
    Load the dry bean data from the ARFF file and return it as a pandas DataFrame.
    """
    # Load the ARFF file
    with open(dry_bean_data_path, "r") as f:
        data = arff.load(f)
    return pd.DataFrame(data["data"], columns=[attr[0] for attr in data["attributes"]])

In [14]:
data = load_dry_bean_data()

In [ ]:
load_bean_data_code_writer = student_a

start_time_ld = now()
data = load_dry_bean_data()
end_time_ld = now()

display(data.head())

#############################################
# Documentation
#############################################

# Now document the raw data and the loaded data using appropriate ontologies.

# Always add these triples for every activity to define the executor!
ld_ass_uuid_executor = "d040a1d1-84ed-4f2c-b206-073f10eb37f8"  # Generate once
load_dry_bean_data_executor = [
    f":load_dry_bean_data prov:qualifiedAssociation :{ld_ass_uuid_executor} .",
    f":{ld_ass_uuid_executor} prov:agent :{executed_by} .",
    f":{ld_ass_uuid_executor} rdf:type prov:Association .",
    f":{ld_ass_uuid_executor} prov:hadRole :{code_executor_role} .",
]
engine.insert(load_dry_bean_data_executor, prefixes=prefixes)

ld_ass_uuid_writer = "700c15df-b8c4-462b-8869-3227bc6a5b78"  # Generate once
ld_report = """
Load all dry bean data. 
"""
load_dry_bean_data_activity = [
    ":load_dry_bean_data rdf:type prov:Activity .",
    ":load_dry_bean_data sc:isPartOf :data_understanding_phase .",
    ":load_dry_bean_data rdfs:comment 'Data Understanding' .",
    f':load_dry_bean_data rdfs:comment """{ld_report}""" .',
    f':load_dry_bean_data prov:startedAtTime "{start_time_ld}"^^xsd:dateTime .',
    f':load_dry_bean_data prov:endedAtTime "{end_time_ld}"^^xsd:dateTime .',
    f":load_dry_bean_data prov:qualifiedAssociation :{ld_ass_uuid_writer} .",
    f":{ld_ass_uuid_writer} prov:agent :{load_bean_data_code_writer} .",
    f":{ld_ass_uuid_writer} rdf:type prov:Association .",
    f":{ld_ass_uuid_writer} prov:hadRole :{code_writer_role} .",
    # INPUT of activity
    ":load_dry_bean_data prov:used :raw_data .",
    ":load_dry_bean_data prov:used :raw_data_path .",
    ":raw_data rdf:type prov:Entity .",
    ":raw_data_path rdf:type prov:Entity .",
    ":raw_data prov:wasDerivedFrom :raw_data_path .",
    # OUTPUT of activity
    ":data rdf:type prov:Entity .",
    ":data prov:wasGeneratedBy :load_dry_bean_data .",
    ":data prov:wasDerivedFrom :raw_data .",
]
engine.insert(load_dry_bean_data_activity, prefixes=prefixes)

# Further descibe the raw data using Croissant
raw_data_triples = [
    ":raw_data rdf:type sc:Dataset .",
    ":raw_data sc:name 'Dry Bean Dataset' .",
    ":raw_data sc:description 'Morphological features describing seven dry bean classes.' .",
    ":raw_data sc:licence 'https://creativecommons.org/licenses/by/4.0/legalcode' .",
    ":raw_data sc:url 'https://doi.org/10.24432/C50S4B' .",
    ":raw_data sc:citeAs '@misc{dry_bean_602,title={{Dry Bean}}, year={2020}, howpublished={UCI Machine Learning Repository}, note={{DOI: https:/doi.org/10.24432/C50S4B}}' .",
    # Continue with futher information about the dataset...
    ":dry_bean_arff rdf:type cr:FileObject .",
    ":dry_bean_arff sc:name 'dry_bean.arff' .",
    ":dry_bean_arff sc:encodingFormat 'text/arff' .",
    ":raw_data sc:distribution :dry_bean_arff .",
    # Continue with further information about the distribution...
    ":raw_recordset rdf:type cr:RecordSet .",
    ":raw_recordset sc:name 'Table of dry bean data' .",
    ":raw_recordset cr:source :dry_bean_arff .",
    ":raw_data cr:recordSet :raw_recordset .",
    # Continue with further information about the recordset...
    ":raw_recordset cr:field :field_area .",
    ":field_area rdf:type cr:Field .",
    ":field_area sc:name 'Area' .",
    ":field_area qudt:abbreviation 'A' .",
    ":field_area sc:description 'The area of a bean zone and the number of pixels within its boundaries.' .",
    ":field_area qudt:Unit unit:PIXEL .",
    ":field_area cr:dataType xsd:integer .",
    ###
    ":raw_recordset cr:field :field_perimeter .",
    ":field_perimeter rdf:type cr:Field .",
    ":field_perimeter sc:name 'Perimeter' .",
    ":field_perimeter qudt:abbreviation 'P' .",
    ":field_perimeter sc:description 'Bean circumference is defined as the length of its border.' .",
    ":field_perimeter cr:dataType xsd:double .",
    ###
    ":raw_recordset cr:field :field_major_axis_length .",
    ":field_major_axis_length rdf:type cr:Field .",
    ":field_major_axis_length sc:name 'Major axis length' .",
    ":field_major_axis_length qudt:abbreviation 'L' .",
    ":field_major_axis_length sc:description 'The distance between the ends of the longest line that can be drawn from a bean.' .",
    ":field_major_axis_length cr:dataType xsd:double .",
    ###
    ":raw_recordset cr:field :field_minor_axis_length .",
    ":field_minor_axis_length rdf:type cr:Field .",
    ":field_minor_axis_length sc:name 'Minor axis length' .",
    ":field_minor_axis_length qudt:abbreviation 'l' .",
    ":field_minor_axis_length sc:description 'The distance of the longest line that can be drawn from the bean while standing perpendicular to the main axis.' .",
    ":field_minor_axis_length cr:dataType xsd:double .",
    ###
    ":raw_recordset cr:field :field_aspect_ratio .",
    ":field_aspect_ratio rdf:type cr:Field .",
    ":field_aspect_ratio sc:name 'Aspect ratio' .",
    ":field_aspect_ratio qudt:abbreviation 'K' .",
    ":field_aspect_ratio sc:description 'Defines the relationship between the major and minor axis lengths.' .",
    ":field_aspect_ratio qudt:expression 'K = L / l' .",
    # ":field_aspect_ratio qudt:relation :field_major_axis_length, :field_minor_axis_length .",
    ":field_aspect_ratio cr:dataType xsd:double .",
    ###
    ":raw_recordset cr:field :field_eccentricity .",
    ":field_eccentricity rdf:type cr:Field .",
    ":field_eccentricity sc:name 'Eccentricity' .",
    ":field_eccentricity qudt:abbreviation 'Ec' .",
    ":field_eccentricity sc:description 'Eccentricity of the ellipse having the same moments as the region.' .",
    ":field_eccentricity cr:dataType xsd:double .",
    ###
    ":raw_recordset cr:field :field_convex_area .",
    ":field_convex_area rdf:type cr:Field .",
    ":field_convex_area sc:name 'Convex area' .",
    ":field_convex_area qudt:abbreviation 'C' .",
    ":field_convex_area sc:description 'Number of pixels in the smallest convex polygon that can contain the area of a bean seed.' .",
    ":field_convex_area cr:dataType xsd:integer .",
    ###
    ":raw_recordset cr:field :field_equivalent_diameter .",
    ":field_equivalent_diameter rdf:type cr:Field .",
    ":field_equivalent_diameter sc:name 'Equivalent diameter' .",
    ":field_equivalent_diameter qudt:abbreviation 'Ed' .",
    ":field_equivalent_diameter sc:description 'The diameter of a circle having the same area as a bean seed area.' .",
    ":field_equivalent_diameter qudt:expression 'Ed = 2 * sqrt(A / pi)' .",
    # ":field_equivalent_diameter qudt:relation :field_area .",
    ":field_equivalent_diameter cr:dataType xsd:double .",
    ###
    ":raw_recordset cr:field :field_extent .",
    ":field_extent rdf:type cr:Field .",
    ":field_extent sc:name 'Extent' .",
    ":field_extent qudt:abbreviation 'Ex' .",
    ":field_extent sc:description 'The ratio of the pixels in the bounding box to the bean area.' .",
    ":field_extent qudt:expression 'Ex = A / Area of bounding rectangle' .",
    # ":field_extent qudt:relation :field_area .",
    ":field_extent cr:dataType xsd:double .",
    ###
    ":raw_recordset cr:field :field_solidity .",
    ":field_solidity rdf:type cr:Field .",
    ":field_solidity sc:name 'Solidity' .",
    ":field_solidity sc:alternateName 'Convexity' .",
    ":field_solidity qudt:abbreviation 'S' .",
    ":field_solidity sc:description 'The ratio of the pixels in the convex shell to those found in beans.' .",
    ":field_solidity qudt:expression 'S = A / C' .",
    # ":field_solidity qudt:relation :field_area, :field_convex_area .",
    ":field_solidity cr:dataType xsd:double .",
    ###
    ":raw_recordset cr:field :field_roundness .",
    ":field_roundness rdf:type cr:Field .",
    ":field_roundness sc:name 'Roundness' .",
    ":field_roundness qudt:abbreviation 'R' .",
    ":field_roundness sc:description 'Ratio of area over the squared of the perimeter' .",
    ":field_roundness qudt:expression 'R = 4 * pi * A / P^2' .",
    # ":field_roundness qudt:relation :field_area, :field_perimeter .",
    ":field_roundness cr:dataType xsd:double .",
    ###
    ":raw_recordset cr:field :field_compactness .",
    ":field_compactness rdf:type cr:Field .",
    ":field_compactness sc:name 'Compactness' .",
    ":field_compactness qudt:abbreviation 'CO' .",
    ":field_compactness sc:description 'Measures the roundness of an object' .",
    ":field_compactness qudt:expression 'CO = Ed / P' .",
    # ":field_compactness qudt:relation :field_equivalent_diameter, :field_perimeter .",
    ":field_compactness cr:dataType xsd:double .",
    ###
    ":raw_recordset cr:field :field_shpf1 .",
    ":field_shpf1 rdf:type cr:Field .",
    ":field_shpf1 sc:name 'Shape Factor 1' .",
    # ":field_shpf1 sc:description 'This is a date field of...' .",
    ":field_shpf1 qudt:expression '= L / A' .",
    # ":field_shpf1 qudt:relation :field_major_axis_length, :field_area .",
    ":field_shpf1 cr:dataType xsd:double .",
    ###
    ":raw_recordset cr:field :field_shpf2 .",
    ":field_shpf2 rdf:type cr:Field .",
    ":field_shpf2 sc:name 'Shape Factor 2' .",
    # ":field_shpf2 sc:description 'This is a date field of...' .",
    ":field_shpf2 qudt:expression '= l / A' .",
    # ":field_shpf2 qudt:relation :field_minor_axis_length, :field_area .",
    ":field_shpf2 cr:dataType xsd:double .",
    ###
    ":raw_recordset cr:field :field_shpf3 .",
    ":field_shpf3 rdf:type cr:Field .",
    ":field_shpf3 sc:name 'Shape Factor 3' .",
    # ":field_shpf3 sc:description 'This is a date field of...' .",
    ":field_shpf3 qudt:expression '= A / (L^2 / 4 * pi)' .",
    # ":field_shpf3 qudt:relation :field_area, :field_major_axis_length .",
    ":field_shpf3 cr:dataType xsd:double .",
    ###
    ":raw_recordset cr:field :field_shpf4 .",
    ":field_shpf4 rdf:type cr:Field .",
    ":field_shpf4 sc:name 'Shape Factor 4' .",
    # ":field_shpf4 sc:description 'This is a date field of...' .",
    ":field_shpf4 qudt:expression '= A / (L / 2 * l / 2 * pi)' .",
    # ":field_shpf4 qudt:relation :field_area, :field_major_axis_length, :field_minor_axis_length .",
    ":field_shpf4 cr:dataType xsd:double .",
    ###
    ":raw_recordset cr:field :field_class .",
    ":field_class rdf:type cr:Field .",
    ":field_class sc:name 'Class' .",
    ":field_class sc:description 'The bean variety' .",
    ":field_class cr:dataType xsd:string .",
]
engine.insert(raw_data_triples, prefixes=prefixes)

# Also the output of the load activity is a dataset that can be described with Croissant
data_triples = [
    ":data rdf:type sc:Dataset .",
    # ...
    ":recordset rdf:type cr:RecordSet .",
    # ...
    ":data cr:recordSet :recordset .",
    # ...
    # Fields can also be reused
    ":recordset cr:field :field_area .",
    ":recordset cr:field :field_perimeter .",
    ":recordset cr:field :field_major_axis_length .",
    ":recordset cr:field :field_minor_axis_length .",
    ":recordset cr:field :field_aspect_ratio .",
    ":recordset cr:field :field_eccentricity .",
    ":recordset cr:field :field_convex_area .",
    ":recordset cr:field :field_equivalent_diameter .",
    ":recordset cr:field :field_extent .",
    ":recordset cr:field :field_solidity .",
    ":recordset cr:field :field_roundness .",
    ":recordset cr:field :field_compactness .",
    ":recordset cr:field :field_shpf1 .",
    ":recordset cr:field :field_shpf2 .",
    ":recordset cr:field :field_shpf3 .",
    ":recordset cr:field :field_shpf4 .",
    ":recordset cr:field :field_class .",
]
engine.insert(data_triples, prefixes=prefixes)


In [16]:
def check_missing_values(df: pd.DataFrame) -> pd.Series:
    """
    Check for missing values in the DataFrame and return a Series with the count of missing values per column.
    """
    return df.isnull().sum()

In [ ]:
check_missing_values(data)

**Continue with other tasks of the Data Understanding phase such as checking the distribution, skewness, plausibility of values, etc...**

In [18]:
def class_imbalance(df: pd.DataFrame):
    """
    Visualizes the class distribution in the dataset.
    """
    plt.figure(figsize=(10, 6))
    sns.countplot(x="Class", data=df)
    plt.title("Class Distribution")
    plt.show()

In [ ]:
class_imbalance(data)

In [ ]:
def plot_correlation_matrix(df: pd.DataFrame):
    """
    For all numerical values, plots correlation matrix for the given DataFrame.
    """
    corr = df.select_dtypes(include=[np.number]).corr()
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, cmap="vlag", annot=True, fmt=".2f")
    plt.title("Correlation Matrix")
    plt.xticks(ticks=np.arange(len(corr.columns)), labels=corr.columns, rotation=90)
    plt.yticks(ticks=np.arange(len(corr.index)), labels=corr.index)
    plt.show()


In [ ]:
plot_correlation_matrix(data)

In [ ]:
def histograms(df: pd.DataFrame):
    """
    Just simple histograms for all numerical columns in the DataFrame.
    """
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols].hist(bins=30, figsize=(15, 10))
    plt.tight_layout()
    plt.show()


In [ ]:
histograms(data)

In [ ]:
def box_plots_per_class(df: pd.DataFrame):
    """
    Shows box plots in 2x1 grids for each numerical feature per class.
    """
    num_cols = df.select_dtypes(include=[np.number]).columns
    class_col = "Class"
    for col in num_cols:
        plt.figure(figsize=(10, 6))
        sns.boxplot(y=class_col, x=col, data=df)
        plt.title(f"Box Plot of {col} by {class_col}")
        plt.grid()
        plt.show()


def violin_plots_per_class(df: pd.DataFrame):
    """
    Shows violin plots in 2x1 grids for each numerical feature per class.
    """
    num_cols = df.select_dtypes(include=[np.number]).columns
    class_col = "Class"
    for col in num_cols:
        plt.figure(figsize=(10, 6))
        sns.violinplot(y=class_col, x=col, data=df)
        plt.title(f"Violin Plot of {col} by {class_col}")
        plt.grid()
        plt.show()

In [ ]:
box_plots_per_class(data)
# violin_plots_per_class(data)

In [ ]:
def brushed_histograms(df: pd.DataFrame, class_col: str):
    """
    Plots stacked bar histograms for all numerical columns in the DataFrame.
    Each bar is stacked by the 'Class' column, with a different color for each class.
    """
    num_cols = df.select_dtypes(include=[np.number]).columns
    classes = df[class_col].unique()
    colors = sns.color_palette("tab10", len(classes))

    for col in num_cols:
        # Prepare data for stacking
        data_per_class = [df[df[class_col] == cls][col].dropna() for cls in classes]
        plt.figure(figsize=(10, 6))
        plt.hist(
            data_per_class,
            bins=50,
            stacked=True,
            color=colors,
            label=classes,
            edgecolor="black",
        )
        plt.title(f"Stacked Histogram of {col} by {class_col}")
        plt.xlabel(col)
        plt.ylabel("Count")
        plt.legend(title=class_col)
        plt.tight_layout()
        plt.show()


def normalized_stacked_histograms(df: pd.DataFrame, class_col: str, bins=20):
    """
    Plots normalized stacked histograms for all numerical columns in the DataFrame.
    Each bin sums to 100% (relative frequency), and each stack shows the class composition per bin.
    """

    num_cols = df.select_dtypes(include=[np.number]).columns
    classes = df[class_col].unique()
    colors = sns.color_palette("tab10", len(classes))

    for col in num_cols:
        # Compute bin edges
        values = df[col].dropna()
        bin_edges = np.histogram_bin_edges(values, bins=bins)
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

        # Compute histogram for each class
        hist_per_class = []
        for cls in classes:
            cls_values = df[df[class_col] == cls][col].dropna()
            hist, _ = np.histogram(cls_values, bins=bin_edges)
            hist_per_class.append(hist)
        hist_per_class = np.array(hist_per_class)  # shape: (n_classes, n_bins)

        # Normalize so each bin sums to 1 (100%)
        bin_sums = hist_per_class.sum(axis=0)
        # Avoid division by zero
        bin_sums[bin_sums == 0] = 1
        hist_norm = hist_per_class / bin_sums

        # Plot
        plt.figure(figsize=(10, 6))
        bottom = np.zeros(hist_norm.shape[1])
        for idx, cls in enumerate(classes):
            plt.bar(
                bin_centers,
                hist_norm[idx],
                width=np.diff(bin_edges),
                bottom=bottom,
                color=colors[idx],
                label=str(cls),
                edgecolor="black",
                align="center",
            )
            bottom += hist_norm[idx]
        plt.title(f"Normalized Stacked Histogram of {col} by {class_col}")
        plt.xlabel(col)
        plt.ylabel("Proportion per bin")
        plt.ylim(0, 1)
        plt.legend(title=class_col)
        plt.tight_layout()
        plt.show()

In [ ]:
# brushed_histograms(data, class_col="Class")
normalized_stacked_histograms(data, class_col="Class", bins=50)

In [ ]:
from matplotlib.patches import Ellipse
from matplotlib.collections import PatchCollection


def spilled_beans(df: pd.DataFrame, x_col: str, y_col: str):
    """
    Fast scatter plot: each bean is an ellipse, colored by class.
    Uses PatchCollection for performance.
    """
    classes = df["Class"].unique()
    colors = sns.color_palette("tab10", len(classes))
    fig, ax = plt.subplots(figsize=(10, 8))

    for idx, cls in enumerate(classes):
        class_data = df[df["Class"] == cls]
        ellipses = [
            Ellipse(
                (row[x_col], row[y_col]),
                width=row["MinorAxisLength"] / 500,
                height=row["MajorAxisLength"] / 500,
                angle=np.random.uniform(0, 360),
            )
            for _, row in class_data.iterrows()
        ]
        if ellipses:
            collection = PatchCollection(
                ellipses,
                facecolor=colors[idx],
                alpha=0.01,
                edgecolor="none",
                label=str(cls),
            )
            ax.add_collection(collection)

    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.legend(title="Class")
    plt.tight_layout()
    plt.show()

In [ ]:
spilled_beans(data, x_col="Extent", y_col="ConvexArea")

In [ ]:
check_outliers_code_writer = student_a


start_time_co = now()


def check_outliers(data: pd.DataFrame, threshold: float = 2.2) -> dict:
    """
    Detects outliers using a z-score based method.

    Parameters:
        data (pd.DataFrame): Input dataset
        threshold (float): Z-score threshold for outlier detection

    Returns:
        dict: Mapping from attribute name to list of row indices
              that were identified as outliers
    """

    numeric_df = data.select_dtypes(include=[np.number])
    outliers_report = {}

    for col in numeric_df.columns:
        col_values = numeric_df[col]

        mean = col_values.mean()
        std = col_values.std()

        # avoid division by zero
        if std == 0 or np.isnan(std):
            continue

        z_scores = (col_values - mean) / std
        outlier_indices = col_values.index[np.abs(z_scores) > threshold].tolist()

        if outlier_indices:
            outliers_report[col] = outlier_indices

    return outliers_report


start_time_ho = now()
outliers_report = check_outliers(data, threshold=2.2)
end_time_ho = now()

#############################################
# Documentation
#############################################
# There are three steps involved in this process:
# 1. activity creates a figure, report etc. => in this case a report
# 2. activity inspects the outcome and derives decisions => in this case to remove the outliers that were found
# 3. activity follows up on the decision by changing the data => will be done in the data preparation phase

# 1. Activity: Checking for outliers and creating the report
co_ass_uuid_executor = "86b05086-70a3-4eac-b257-97c9680ceb00"
check_outliers_executor = [
    f":check_outliers prov:qualifiedAssociation :{co_ass_uuid_executor} .",
    f":{co_ass_uuid_executor} prov:agent :{executed_by} .",
    f":{co_ass_uuid_executor} rdf:type prov:Association .",
    f":{co_ass_uuid_executor} prov:hadRole :{code_executor_role} .",
]
engine.insert(check_outliers_executor, prefixes=prefixes)

co_ass_uuid_writer = "6f26e066-050a-42c2-9a65-7b5d8a0f8416"
co_comment = (
    "A z-score based method (threshold = 2.2) was used to identify extreme values in "
    "numeric attributes. This step was exploratory and did not directly determine "
    "which data instances should be removed."
)
check_outliers_activity = [
    ":check_outliers rdf:type prov:Activity .",
    ":check_outliers sc:isPartOf :data_understanding_phase .",
    ":check_outliers rdfs:comment 'Data Understanding' .",
    f':check_outliers rdfs:comment """{co_comment}""" .',
    # f':check_outliers prov:startedAtTime "{start_time_co}"^^xsd:dateTime .',
    # f':check_outliers prov:endedAtTime "{end_time_co}"^^xsd:dateTime .',
    f":check_outliers prov:qualifiedAssociation :{co_ass_uuid_writer} .",
    f":{co_ass_uuid_writer} prov:agent :{check_outliers_code_writer} .",
    f":{co_ass_uuid_writer} rdf:type prov:Association .",
    f":{co_ass_uuid_writer} prov:hadRole :{code_writer_role} .",
    ":check_outliers prov:used :data .",
    ":outlier_report rdf:type prov:Entity .",
    f':outlier_report rdfs:comment """{json.dumps(outliers_report, indent=2)}""" .',
    ":outlier_report prov:wasGeneratedBy :check_outliers .",
    # ...
]
engine.insert(check_outliers_activity, prefixes=prefixes)

# 2. Activity: Inspecting the report and taking a decision on what to do
ior_ass_uuid_executor = "02c33e8a-5f5f-44ad-bce9-f05e629d9056"
ior_comment = (
    "After inspecting the boxplots, one extreme instance was identified in the "
    "ShapeFactor4 attribute for the Seker class. This instance was considered an "
    "outlier and removed, while all other data points were kept."
)

inspect_outlier_report_activity = [
    ":inspect_outlier_report rdf:type prov:Activity .",
    ":inspect_outlier_report rdfs:comment 'Data Understanding' .",
    f':inspect_outlier_report rdfs:comment """{ior_comment}""" .',
    f':inspect_outlier_report prov:startedAtTime "{start_time_ho}"^^xsd:dateTime .',
    f':inspect_outlier_report prov:endedAtTime "{end_time_ho}"^^xsd:dateTime .',
    f":inspect_outlier_report prov:qualifiedAssociation :{ior_ass_uuid_executor} .",
    f":{ior_ass_uuid_executor} prov:agent :{check_outliers_code_writer} .",
    f":{ior_ass_uuid_executor} rdf:type prov:Association .",
    f":{ior_ass_uuid_executor} prov:hadRole :{code_executor_role} .",
    ":inspect_outlier_report prov:used :outlier_report .",
    ":outlier_decision rdf:type prov:Entity .",
    ':outlier_decision rdfs:comment """Removing all outliers for demonstration purposes.""" .',  # TODO
    ":outlier_decision prov:wasGeneratedBy :inspect_outlier_report .",
    # ...
]
engine.insert(inspect_outlier_report_activity, prefixes=prefixes)


In [ ]:
def check_statistical_properties(data: pd.DataFrame) -> str:
    """
    Computes statistical properties and correlations of the dataset and
    returns a textual interpretation suitable for rdfs:comment logging.
    """

    numeric_df = data.select_dtypes(include=[np.number])

    comments = []

    # Basic statistics
    desc = numeric_df.describe()

    comments.append(
        f"The dataset contains {len(data)} records and {numeric_df.shape[1]} numeric attributes."
    )

    for col in numeric_df.columns:
        comments.append(
            f"Attribute '{col}' has mean={desc.loc['mean', col]:.2f}, "
            f"std={desc.loc['std', col]:.2f}, "
            f"min={desc.loc['min', col]:.2f}, "
            f"max={desc.loc['max', col]:.2f}."
        )

    # Correlations
    corr = numeric_df.corr(method="pearson")

    strong_corrs = []
    for i in range(len(corr.columns)):
        for j in range(i + 1, len(corr.columns)):
            val = corr.iloc[i, j]
            if abs(val) >= 0.7:
                strong_corrs.append(
                    f"{corr.columns[i]} and {corr.columns[j]} (r={val:.2f})"
                )

    if strong_corrs:
        comments.append(
            "Strong correlations for attribute pairs: " + "; ".join(strong_corrs) + "."
        )
    else:
        comments.append("No strong correlations (|r| ≥ 0.7).")

    return " ".join(comments)

In [ ]:
statistical_properties_code_writer = student_a

start_time_sd = now()
statistical_properties_report = check_statistical_properties(data)
end_time_sd = now()

# 1. Activity: Checking statistical properties of the dataset
csd_ass_uuid_executor = "2144636f-fccc-439c-9827-19b556921051"

check_statistical_properties_executor = [
    f":check_statistical_properties prov:qualifiedAssociation :{csd_ass_uuid_executor} .",
    f":{csd_ass_uuid_executor} prov:agent :{executed_by} .",
    f":{csd_ass_uuid_executor} rdf:type prov:Association .",
    f":{csd_ass_uuid_executor} prov:hadRole :{code_executor_role} .",
]

engine.insert(check_statistical_properties_executor, prefixes=prefixes)

csd_ass_uuid_writer = "91808747-b164-46d7-8472-cc43969911d6"

sd_comment = statistical_properties_report
sd_comment_safe = (
    sd_comment.replace("\n", " ")
    .replace("\r", " ")
    .replace("\t", " ")
    .replace('"', "'")
)

statistical_properties_activity = [
    ":stat_properties rdf:type prov:Activity .",
    ":stat_properties sc:isPartOf :data_understanding_phase .",
    ":stat_properties rdfs:comment 'Data Understanding' .",
    f':stat_properties rdfs:comment "{sd_comment_safe}" .',
    f':stat_properties prov:startedAtTime "{start_time_sd}"^^xsd:dateTime .',
    f':stat_properties prov:endedAtTime "{end_time_sd}"^^xsd:dateTime .',
    f":stat_properties prov:qualifiedAssociation :{csd_ass_uuid_writer} .",
    f":{csd_ass_uuid_writer} prov:agent :{statistical_properties_code_writer} .",
    f":{csd_ass_uuid_writer} rdf:type prov:Association .",
    f":{csd_ass_uuid_writer} prov:hadRole :{code_writer_role} .",
    ":stat_properties prov:used :data .",
]

engine.insert(statistical_properties_activity, prefixes=prefixes)

In [ ]:
def check_data_quality(data: pd.DataFrame) -> str:
    """
    Analyzes data quality aspects and returns an interpretation
    suitable for rdfs:comment logging.
    """

    comments = []
    n_rows = len(data)

    # Missing values
    missing = data.isnull().sum()
    missing_cols = missing[missing > 0]

    if not missing_cols.empty:
        for col, cnt in missing_cols.items():
            perc = (cnt / n_rows) * 100
            comments.append(
                f"Attribute '{col}' contains {cnt} missing values ({perc:.2f}%), "
                "This can affect statistical reliability."
            )
    else:
        comments.append("No missing values were detected in the dataset.")

    # Uneven distributions
    numeric_df = data.select_dtypes(include=[np.number])
    for col in numeric_df.columns:
        skew = numeric_df[col].skew()
        if abs(skew) > 1:
            comments.append(
                f"Attribute '{col}' shows a strongly skewed distribution (skewness={skew:.2f}), "
                "indicating an uneven distribution of values."
            )

    # Outliers
    outliers_report = check_outliers(data, threshold=2.2)

    if isinstance(outliers_report, dict) and outliers_report:
        outliers_count = sum(len(v) for v in outliers_report.values())
        affected_attributes = ", ".join(outliers_report.keys())

        comments.append(
            f"{outliers_count} records were identified as potential outliers using the "
            f"existing z-score based outlier detection method (threshold = 2.2). "
            f"Outliers were detected in the following attributes: {affected_attributes}. "
            "These extreme values may represent rare but valid cases and should be "
            "handled carefully to avoid reducing the representation of uncommon patterns."
        )
    else:
        comments.append(
            "No outliers were identified using the existing z-score based outlier detection "
            "method, indicating the absence of extreme values according to the applied criteria."
        )

    return " ".join(comments)

    rows_before_outliers = len(data)
    data = handle_outliers(data, None)
    rows_after_outliers = len(data)

    outliers_removed = rows_before_outliers - rows_after_outliers
    if outliers_removed > 0:
        comments.append(
            f"{outliers_removed} records were identified as outliers and removed using the "
            "existing outlier handling function defined earlier in the notebook. "
            "This improves data robustness but may reduce the representation of extreme cases."
        )
    else:
        comments.append(
            "No records were removed during the outlier handling step, indicating the absence "
            "of extreme values according to the applied criteria."
        )

    # Plausibility
    if (numeric_df < 0).any().any():
        comments.append(
            "Some numeric attributes contain negative values. "
            "Depending on attribute semantics, these values may be implausible."
        )
    else:
        comments.append("All numeric values are non-negative and appear plausible ")

    comments.append(
        "No explicit information about prior data cleansing or preprocessing "
        "steps is available in the dataset provenance."
    )

    return " ".join(comments)

In [ ]:
data_quality_code_writer = student_a

start_time_dq = now()
data_quality_report = check_data_quality(data)
end_time_dq = now()

dq_ass_uuid_writer = "7c642b03-1d46-4fdd-8bca-a8fabe86445d"

dq_comment_safe = (
    data_quality_report.replace("\n", " ")
    .replace("\r", " ")
    .replace("\t", " ")
    .replace('"', "'")
)

data_quality_activity = [
    ":data_quality rdf:type prov:Activity .",
    ":data_quality sc:isPartOf :data_understanding_phase .",
    ":data_quality rdfs:comment 'Data Quality Assessment' .",
    f':data_quality rdfs:comment "{dq_comment_safe}" .',
    f':data_quality prov:startedAtTime "{start_time_dq}"^^xsd:dateTime .',
    f':data_quality prov:endedAtTime "{end_time_dq}"^^xsd:dateTime .',
    f":data_quality prov:qualifiedAssociation :{dq_ass_uuid_writer} .",
    f":{dq_ass_uuid_writer} prov:agent :{data_quality_code_writer} .",
    f":{dq_ass_uuid_writer} rdf:type prov:Association .",
    f":{dq_ass_uuid_writer} prov:hadRole :{code_writer_role} .",
    ":data_quality prov:used :data .",
]

engine.insert(data_quality_activity, prefixes=prefixes)


In [ ]:
def visual_exploration_and_hypotheses(data: pd.DataFrame) -> str:
    """
    Performs visual exploration of data properties and returns
    a textual interpretation and hypotheses suitable for rdfs:comment.
    """

    comments = []

    numeric_df = data.select_dtypes(include=[np.number])

    # Distributions
    for col in numeric_df.columns[:4]:  # limit to avoid overload
        plt.figure()
        numeric_df[col].hist(bins=30)
        plt.title(f"Distribution of {col}")
        plt.xlabel(col)
        plt.ylabel("Frequency")
        plt.show()

        comments.append(
            f"The distribution of attribute '{col}' was visually inspected using a histogram, "
            "revealing its spread and central tendency."
        )

    # Correlation heatmap
    corr = numeric_df.corr()

    plt.figure(figsize=(8, 6))
    plt.imshow(corr, cmap="coolwarm", interpolation="nearest")
    plt.colorbar()
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.columns)), corr.columns)
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.show()

    comments.append(
        "A correlation heatmap was generated to visually assess relationships between numeric "
        "attributes, supporting hypotheses about linear dependencies."
    )

    # Hypotheses
    comments.append(
        "Based on the visual exploration, it is hypothesized that size-related attributes "
        "exhibit strong positive relationships, while shape-related attributes vary more independently."
    )

    return " ".join(comments)


In [ ]:
visual_exploration_code_writer = student_a

start_time_ve = now()
visual_exploration_report = visual_exploration_and_hypotheses(data)
end_time_ve = now()

ve_ass_uuid_writer = "eceed4f7-ddac-4987-8ef3-a796f05e4d6e"

ve_comment_safe = (
    visual_exploration_report.replace("\n", " ")
    .replace("\r", " ")
    .replace("\t", " ")
    .replace('"', "'")
)

visual_exploration_activity = [
    ":visual_exploration rdf:type prov:Activity .",
    ":visual_exploration sc:isPartOf :data_understanding_phase .",
    # ":visual_exploration rdfs:comment 'Visual Exploration and Hypotheses' .",
    f':visual_exploration rdfs:comment "{ve_comment_safe}" .',
    f':visual_exploration prov:startedAtTime "{start_time_ve}"^^xsd:dateTime .',
    f':visual_exploration prov:endedAtTime "{end_time_ve}"^^xsd:dateTime .',
    f":visual_exploration prov:qualifiedAssociation :{ve_ass_uuid_writer} .",
    f":{ve_ass_uuid_writer} prov:agent :{visual_exploration_code_writer} .",
    f":{ve_ass_uuid_writer} rdf:type prov:Association .",
    f":{ve_ass_uuid_writer} prov:hadRole :{code_writer_role} .",
    ":visual_exploration prov:used :data .",
]

engine.insert(visual_exploration_activity, prefixes=prefixes)

In [ ]:
ethical_bias_comment = """
Ethical sensitivity, minority groups, and distributional bias:
The dataset has numeric geometric and morphological attributes that come from object measurements.
 No attributes have personal, demographic, socio-economic, or
identifiable information. So the dataset does not contain ethically sensitive attributes.
 However, strongly skewed distributions for several
attributes indicate not well represented regions of the feature space, showing rare or
extreme geometric configurations. These may
lead to representation bias in modeling tasks, where common patterns dominate model behavior.

Potential risks, bias sources, and expert questions:
There are some potential risks. Strong skewness and extreme values show representation
bias against uncommon shapes and sizes. Strong correlations among size-related attributes show
possible sampling bias toward objects on similar scale. The lack of detailed data
provenance questions data acquisition, selection criteria, and prior
preprocessing steps. To better assess bias and data quality risks, these questions may help: Does the dataset reflect real-world variability of object
shapes and sizes? Were certain object types or ranges excluded during collection? Are extreme values
rare in reality or under-represented due to sampling or preprocessing decisions?

Required data preparation actions:
Data preparation should handle skewed attributes, preserve rare cases, manage correlated features, and use evaluation metrics that account for imbalance. Improving data provenance would increase transparency.
"""

ethical_bias_comment_safe = (
    ethical_bias_comment.replace("\n", " ")
    .replace("\r", " ")
    .replace("\t", " ")
    .replace('"', "'")
)

ethical_bias_activity = [
    ":ethical_bias_assessment rdf:type prov:Activity .",
    ":ethical_bias_assessment sc:isPartOf :data_understanding_phase .",
    f':ethical_bias_assessment rdfs:comment "{ethical_bias_comment_safe}" .',
]

engine.insert(ethical_bias_activity, prefixes=prefixes)


## Data Preparation

In [20]:
## Each Activity that follows is part of the Data Preparation Phase

data_preparation_phase_executor = [
    ":data_preparation_phase rdf:type prov:Activity .",
    ':data_preparation_phase rdfs:label "Data Preparation Phase" .',
]
engine.insert(data_preparation_phase_executor, prefixes=prefixes)

In [21]:
def handle_outliers(df: pd.DataFrame, outliers_report: dict) -> pd.DataFrame:
    condition = (df["Class"] == "SEKER") & (df["ShapeFactor4"] < 0.95)
    df = df[~condition].reset_index(drop=True)
    return df

In [22]:
data = handle_outliers(data, None)

In [23]:
handle_outliers_code_writer = student_b


start_time_td = now()
data = handle_outliers(data, None)
end_time_td = now()

#############################################
# Documentation
#############################################
# This is the continuation of the example from the Data Understanding phase above.
# There are three steps involved in this process:
# 1. activity creates a figure, report etc. => already done in data understanding phase
# 2. activity inspects the outcome and derives decisions => already done in data understanding phase
# 3. activity follows up on the decision by changing the data => in this case by removing the the outliers that were found

ro_ass_uuid_executor = "7d6cecd2-0cc9-4094-a10b-4a601b76463b"
handle_outliers_executor = [
    f":handle_outliers prov:qualifiedAssociation :{ro_ass_uuid_executor} .",
    f":{ro_ass_uuid_executor} prov:agent :{executed_by} .",
    f":{ro_ass_uuid_executor} rdf:type prov:Association .",
    f":{ro_ass_uuid_executor} prov:hadRole :{code_executor_role} .",
]
engine.insert(handle_outliers_executor, prefixes=prefixes)

td_ass_uuid_writer = "e75185e3-0328-49ba-99da-9003d45b0b9a"
td_comment = """
Removed all instances of the SEKER class where `ShapeFactor4` was below 0.95.
"""
handle_outliers_activity = [
    ":handle_outliers rdf:type prov:Activity .",
    ":handle_outliers sc:isPartOf :data_preparation_phase .",
    ":handle_outliers rdfs:comment 'Data Preparation' .",
    f':handle_outliers rdfs:comment """{td_comment}""" .',
    f':handle_outliers prov:startedAtTime "{start_time_td}"^^xsd:dateTime .',
    f':handle_outliers prov:endedAtTime "{end_time_td}"^^xsd:dateTime .',
    f":handle_outliers prov:qualifiedAssociation :{td_ass_uuid_writer} .",
    f":{td_ass_uuid_writer} prov:agent :{handle_outliers_code_writer} .",
    f":{td_ass_uuid_writer} rdf:type prov:Association .",
    f":{td_ass_uuid_writer} prov:hadRole :{code_writer_role} .",
    ":handle_outliers prov:used :data .",
    ":handle_outliers prov:used :outlier_decision .",
    ":cleaned_data rdf:type prov:Entity .",
    ":cleaned_data prov:wasGeneratedBy :handle_outliers .",
    ":cleaned_data prov:wasDerivedFrom :data .",
]
engine.insert(handle_outliers_activity, prefixes=prefixes)

HTTPError: HTTP Error 504: Gateway Time-out

**Continue with other tasks of the Data Preparation phase such as binning, scaling etc...**

In [31]:
def one_hot_encoding(df: pd.DataFrame) -> pd.DataFrame:
    """
    The class has seven values. We transform the 'Class' column into one-hot encoded columns.
    The original class column is dropped.
    """
    dfc = df.copy()
    dfc = pd.get_dummies(dfc, columns=["Class"], prefix="Class")
    return dfc


def scaling(
    df: pd.DataFrame, strategy: str
) -> tuple[pd.DataFrame, tuple[TransformerMixin, list[str]]]:
    """
    Scaling will only be applied to the columns
    - Area
    - Perimeter
    - MajorAxisLength
    - MinorAxisLength
    - ConvexArea
    - EquivDiameter
    """
    dfc = df.copy()
    cols_to_scale = [
        "Area",
        "Perimeter",
        "MajorAxisLength",
        "MinorAxisLength",
        "ConvexArea",
        "EquivDiameter",
    ]
    if strategy == "min-max":
        scaler = MinMaxScaler()
    elif strategy == "standard":
        scaler = StandardScaler()
    else:
        raise ValueError(f"Unknown scaling strategy: {strategy}")
    dfc[cols_to_scale] = scaler.fit_transform(dfc[cols_to_scale])
    return dfc, (scaler, cols_to_scale)

In [33]:
data_ohe = one_hot_encoding(data)
# Actually not needed

In [ ]:
# Your final transformed dataset should also be documented appropriately using Croissant, SI, etc.

prepared_data_triples = [
    ":prepared_data rdf:type prov:Entity .",
    ":prepared_data prov:wasDerivedFrom :cleaned_data .",
    ":prepared_data rdf:type sc:Dataset .",
    # ....
    ":prepared_data rdf:type sc:Dataset .",
    ":prepared_data sc:name 'Prepared Dry Bean Dataset' .",
    ":prepared_data sc:description 'Morphological features describing seven dry bean classes.' .",
    ":prepared_data sc:licence 'https://creativecommons.org/licenses/by/4.0/legalcode' .",
    # Continue with further information about the distribution...
    ":prepared_recordset rdf:type cr:RecordSet .",
    ":prepared_recordset sc:name 'Table of prepared dry bean data' .",
    ":prepared_recordset cr:source :dry_bean_arff .",
    ":prepared_data cr:recordSet :prepared_recordset .",
    ###
    ":prepared_recordset cr:field :field_class_seker .",
    ":field_class_seker rdf:type cr:Field .",
    ":field_class_seker sc:name 'Class_SEKER' .",
    ":field_class_seker sc:description 'Seker dry bean variety' .",
    ":field_class_seker cr:dataType xsd:boolean .",
    #
    ":prepared_recordset cr:field :field_class_barbunya .",
    ":field_class_barbunya rdf:type cr:Field .",
    ":field_class_barbunya sc:name 'Class_BARBUNYA' .",
    ":field_class_barbunya sc:description 'Barbunya dry bean variety' .",
    ":field_class_barbunya cr:dataType xsd:boolean .",
    #
    ":prepared_recordset cr:field :field_class_bombay .",
    ":field_class_bombay rdf:type cr:Field .",
    ":field_class_bombay sc:name 'Class_BOMBAY' .",
    ":field_class_bombay sc:description 'Bombay dry bean variety' .",
    ":field_class_bombay cr:dataType xsd:boolean .",
    #
    ":prepared_recordset cr:field :field_class_cali .",
    ":field_class_cali rdf:type cr:Field .",
    ":field_class_cali sc:name 'Class_CALI' .",
    ":field_class_cali sc:description 'Cali dry bean variety' .",
    ":field_class_cali cr:dataType xsd:boolean .",
    #
    ":prepared_recordset cr:field :field_class_dermosan .",
    ":field_class_dermosan rdf:type cr:Field .",
    ":field_class_dermosan sc:name 'Class_DERMOSAN' .",
    ":field_class_dermosan sc:description 'Dermosan dry bean variety' .",
    ":field_class_dermosan cr:dataType xsd:boolean .",
    #
    ":prepared_recordset cr:field :field_class_horoz .",
    ":field_class_horoz rdf:type cr:Field .",
    ":field_class_horoz sc:name 'Class_HOROZ' .",
    ":field_class_horoz sc:description 'Horoz dry bean variety' .",
    ":field_class_horoz cr:dataType xsd:boolean .",
    #
    ":prepared_recordset cr:field :field_class_sira .",
    ":field_class_sira rdf:type cr:Field .",
    ":field_class_sira sc:name 'Class_SIRA' .",
    ":field_class_sira sc:description 'Sira dry bean variety' .",
    ":field_class_sira cr:dataType xsd:boolean .",
]
engine.insert(prepared_data_triples, prefixes=prefixes)

In [ ]:
other_preproc_steps_uuid_executor = "5f7b7bc4-bb88-4909-9a38-8c5b2a4a1864"

other_preproc_steps_executor = [
    f":other_preproc_steps prov:qualifiedAssociation :{other_preproc_steps_uuid_executor} .",
    f":{other_preproc_steps_uuid_executor} prov:agent :{executed_by} .",
    f":{other_preproc_steps_uuid_executor} rdf:type prov:Association .",
    f":{other_preproc_steps_uuid_executor} prov:hadRole :{code_executor_role} .",
]

other_prepro_comment = """
We have considered scaling some of the features with highest mean, like the area or the perimeter. However, we consider this scaling (done or not) cannot be performed before train/validation/test splitting due to possible data leakage. Test set instances should never influence how much the train and validation sets are scaled.

Adding more derived attributes does not seem truly necessary, as many of the already exisiting features are already derivations of other measures. So much so, that the four last attributes are simply called "ShapeFactorX".

Additional information that could be interesting to add into the dataset is some form of color information. According to the original paper where this dataset is presented, all dry bean images were first converted into black and white images for feature extraction. At the same time, we know that one of the dry bean varieties, the Barbunya, possesses certain colourful characteristics, that are lost during b/w conversion.
"""

other_preprocessing_steps_activity = [
    ":other_preproc_steps rdf:type prov:Activity .",
    ":other_preproc_steps sc:isPartOf :data_preparation_phase .",
    f':other_preproc_steps rdfs:comment """{other_prepro_comment}""" .',
]

engine.insert(other_preprocessing_steps_activity, prefixes=prefixes)

## Modeling

In [24]:
## Each Activity that follows is part of the Modeling Phase

modeling_phase_executor = [
    ":modeling_phase rdf:type prov:Activity .",
    ':modeling rdfs:label "Modeling Phase" .',
]
engine.insert(modeling_phase_executor, prefixes=prefixes)


In [25]:
model_data_code_writer = student_a

#############################################
# Documentation 4a
#############################################

dma_ass_uuid_writer = "097ea442-7b97-464d-926f-ad0a0882d142"
dma_comment = """
Logistic Regression - fast baseline, linear decision boundary
Decision Tree –  capture non linear relationships but can overfit on its own.
k-Nearest Neighbors - simple, no parameter, but costly at prediction time
Random Forest - non linear, strong performance, robust when correlation

from these algorithms, random forest was selected as the dataset has many correlated attributes and shows skewed distributions. This
algorithm without feature scaling and can robustly catch non linear relationships
"""

identify_data_mining_algorithm_activity = [
    ":define_algorithm rdf:type prov:Activity .",
    ":define_algorithm sc:isPartOf :modeling_phase .",
    f':define_algorithm rdfs:comment """{dma_comment}""" .',
    f":define_algorithm prov:qualifiedAssociation :{dma_ass_uuid_writer} .",
    f":{dma_ass_uuid_writer} prov:agent :{model_data_code_writer} .",
    f":{dma_ass_uuid_writer} rdf:type prov:Association .",
    f":{dma_ass_uuid_writer} prov:hadRole :{code_writer_role} .",
    # example algorithm definition
    ":random_forest_algorithm rdf:type mls:Algorithm .",
    ':random_forest_algorithm rdfs:label "Random Forest Algorithm" .',
    # example implementation
    ":random_forrest_classifier_implementation rdf:type mls:Implementation .",
    ':random_forrest_classifier_implementation rdfs:label "Scikit-learn RandomForestClassifier" .',
    ":random_forrest_classifier_implementation mls:implements :random_forest_algorithm .",
    ":random_forrest_classifier_implementation prov:wasGeneratedBy :define_algorithm .",

    ":accuracy_measure rdf:type mls:EvaluationMeasure .",
    ':accuracy_measure rdfs:label "Accuracy" .',
    ":accuracy_measure prov:wasGeneratedBy :define_algorithm .",

    ":macro_f1_measure rdf:type mls:EvaluationMeasure .",
    ':macro_f1_measure rdfs:label "Macro F1" .',
    ":macro_f1_measure prov:wasGeneratedBy :define_algorithm .",
]
engine.insert(identify_data_mining_algorithm_activity, prefixes=prefixes)

In [28]:
#############################################
# Documentation 4b
#############################################

hp_ass_uuid_writer = "0635254c-37a7-4037-97f2-4e8991aef199"

hp_comment = """
Random Forest Classifier Hyperparameters: n_estimators (number of trees), max_depth, min_samples_split, min_samples_leaf, max_features
bootstrap, and class_weight. For tuning, n_estimators was chosen as the most relevant parameter because it controls the number of
trees and improves stability and performance at the cost of additional computation time.
"""
hp_comment_clean = (
    hp_comment
    .replace("\n", " ")
    .replace("\r", " ")
    .replace("\t", " ")
    .replace('"', "'")
)

identify_hp_activity = [
    ":identify_hyperparameters rdf:type prov:Activity .",
    ":identify_hyperparameters sc:isPartOf :modeling_phase .",
    f':identify_hyperparameters rdfs:comment "{hp_comment_clean}" .',

    f":identify_hyperparameters prov:qualifiedAssociation :{hp_ass_uuid_writer} .",
    f":{hp_ass_uuid_writer} prov:agent :{model_data_code_writer} .",
    f":{hp_ass_uuid_writer} rdf:type prov:Association .",
    f":{hp_ass_uuid_writer} prov:hadRole :{code_writer_role} .",

    # Identified hyperparameters (as entities)
    ":hp_n_estimators rdf:type mls:HyperParameter .",
    ':hp_n_estimators rdfs:label "n_estimators" .',
    ':hp_n_estimators rdfs:comment "Number of trees in the forest" .',
    ":random_forrest_classifier_implementation mls:hasHyperParameter :hp_n_estimators .",
    ":hp_n_estimators prov:wasGeneratedBy :identify_hyperparameters .",

    ":hp_max_depth rdf:type mls:HyperParameter .",
    ':hp_max_depth rdfs:label "max_depth" .',
    ':hp_max_depth rdfs:comment "Maximum depth of each tree" .',
    ":random_forrest_classifier_implementation mls:hasHyperParameter :hp_max_depth .",
    ":hp_max_depth prov:wasGeneratedBy :identify_hyperparameters .",

    ":hp_min_samples_split rdf:type mls:HyperParameter .",
    ':hp_min_samples_split rdfs:label "min_samples_split" .',
    ':hp_min_samples_split rdfs:comment "Minimum num of samples to split an internal node." .',
    ":random_forrest_classifier_implementation mls:hasHyperParameter :hp_min_samples_split .",
    ":hp_min_samples_split prov:wasGeneratedBy :identify_hyperparameters .",

    ":hp_min_samples_leaf rdf:type mls:HyperParameter .",
    ':hp_min_samples_leaf rdfs:label "min_samples_leaf" .',
    ':hp_min_samples_leaf rdfs:comment "Minimum num required to be at a leaf node." .',
    ":random_forrest_classifier_implementation mls:hasHyperParameter :hp_min_samples_leaf .",
    ":hp_min_samples_leaf prov:wasGeneratedBy :identify_hyperparameters .",

    ":hp_max_features rdf:type mls:HyperParameter .",
    ':hp_max_features rdfs:label "max_features" .',
    ':hp_max_features rdfs:comment "Number of features considered when looking for the best split." .',
    ":random_forrest_classifier_implementation mls:hasHyperParameter :hp_max_features .",
    ":hp_max_features prov:wasGeneratedBy :identify_hyperparameters .",

    ":hp_bootstrap rdf:type mls:HyperParameter .",
    ':hp_bootstrap rdfs:label "bootstrap" .',
    ':hp_bootstrap rdfs:comment " are bootstrap samples used when creating trees." .',
    ":random_forrest_classifier_implementation mls:hasHyperParameter :hp_bootstrap .",
    ":hp_bootstrap prov:wasGeneratedBy :identify_hyperparameters .",

    ":hp_class_weight rdf:type mls:HyperParameter .",
    ':hp_class_weight rdfs:label "class_weight" .',
    ':hp_class_weight rdfs:comment "Weights associated with classes to improve imbalance " .',
    ":random_forrest_classifier_implementation mls:hasHyperParameter :hp_class_weight .",
    ":hp_class_weight prov:wasGeneratedBy :identify_hyperparameters .",
]

engine.insert(identify_hp_activity, prefixes=prefixes)


HTTPError: HTTP Error 504: Gateway Time-out

In [41]:
def split_data(X: pd.DataFrame, y: pd.Series, strategy: str):
    seed = 42

    test_size = 0.10
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=test_size, random_state=seed, shuffle=True, stratify=y
    )

    val_size = 0.20
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=val_size, random_state=seed, shuffle=True, stratify=y_train_val
    )

    # Scale ONLY on train, apply to val/test
    X_train_scaled, scale = scaling(X_train, strategy)

    X_val_scaled = X_val.copy()
    X_val_scaled[scale[1]] = scale[0].transform(X_val_scaled[scale[1]])

    X_test_scaled = X_test.copy()
    X_test_scaled[scale[1]] = scale[0].transform(X_test_scaled[scale[1]])

    # return as dataframes with a "Class" column for your later
    train_df = X_train_scaled.copy()
    train_df["Class"] = y_train.values

    val_df = X_val_scaled.copy()
    val_df["Class"] = y_val.values

    test_df = X_test_scaled.copy()
    test_df["Class"] = y_test.values

    return train_df, val_df, test_df


#############################################
# Documentation 4c
#############################################

start_time_ds = now()
train_df, val_df, test_df = split_data(
    data.drop(columns=["Class"]), data["Class"], strategy="min-max"
)
end_time_ds = now()

train_n = len(train_df)
val_n = len(val_df)
test_n = len(test_df)
total_n = train_n + val_n + test_n

split_ass_uuid_writer = "2a79da72-5b87-4d91-b7ab-e52e467edc26"

split_comment = (
    f"The dataset was split reproducibly into training, validation, and test sets with "
    f"{train_n} training, {val_n} validation, and {test_n} test samples by using a fixed random seed equal to 42."
)


# Use prepared dataset
input_dataset = ":prepared_data"

define_split_activity = [
    ":define_data_split rdf:type prov:Activity .",
    ":define_data_split sc:isPartOf :modeling_phase .",
    ':define_data_split rdfs:comment "Train/Validation/Test Split Definition" .',
    f':define_data_split rdfs:comment "{split_comment}" .',
    f':define_data_split prov:startedAtTime "{start_time_ds}"^^xsd:dateTime .',
    f':define_data_split prov:endedAtTime "{end_time_ds}"^^xsd:dateTime .',

    f":define_data_split prov:qualifiedAssociation :{split_ass_uuid_writer} .",
    f":{split_ass_uuid_writer} prov:agent :{model_data_code_writer} .",
    f":{split_ass_uuid_writer} rdf:type prov:Association .",
    f":{split_ass_uuid_writer} prov:hadRole :{code_writer_role} .",

    f":define_data_split prov:used {input_dataset} .",

    # Training Set
    ":training_set rdf:type sc:Dataset .",
    ':training_set rdfs:label "Training Set" .',
    ":training_set prov:wasGeneratedBy :define_data_split .",
    f":training_set prov:wasDerivedFrom {input_dataset} .",
    f':training_set rdfs:comment "Contains {train_n} samples." .',

    # Validation Set
    ":validation_set rdf:type sc:Dataset .",
    ':validation_set rdfs:label "Validation Set" .',
    ":validation_set prov:wasGeneratedBy :define_data_split .",
    f":validation_set prov:wasDerivedFrom {input_dataset} .",
    f':validation_set rdfs:comment "Contains {val_n} samples." .',

    # Test Set
    ":test_set rdf:type sc:Dataset .",
    ':test_set rdfs:label "Test Set" .',
    ":test_set prov:wasGeneratedBy :define_data_split .",
    f":test_set prov:wasDerivedFrom {input_dataset} .",
    f':test_set rdfs:comment "Contains {test_n} samples." .',
]

engine.insert(define_split_activity, prefixes=prefixes)


In [ ]:
def train_and_finetune_model(training_set, validation_set):
    # Split X/y
    X_train = training_set.drop(columns=["Class"])
    y_train = training_set["Class"]
    X_val = validation_set.drop(columns=["Class"])
    y_val = validation_set["Class"]

    # Hyperparameter to tune: n_estimators
    hp_values = [50, 100, 200, 400]

    results = []
    best_n = None
    best_macro_f1 = -1.0

    for n in hp_values:
        model = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
        model.fit(X_train, y_train)

        val_pred = model.predict(X_val)
        val_acc = accuracy_score(y_val, val_pred)
        val_macro_f1 = f1_score(y_val, val_pred, average="macro")

        results.append({
            "n_estimators": n,
            "val_accuracy": float(val_acc),
            "val_macro_f1": float(val_macro_f1)
        })

        if val_macro_f1 > best_macro_f1:
            best_macro_f1 = val_macro_f1
            best_n = n

    # 4e: tuning figure
    xs = [r["n_estimators"] for r in results]
    ys = [r["val_macro_f1"] for r in results]

    plt.figure(figsize=(6, 4))
    plt.plot(xs, ys, marker="o")
    plt.title("Random Forest tuning: n_estimators vs Validation Macro F1")
    plt.xlabel("n_estimators")
    plt.ylabel("Validation Macro F1")
    plt.tight_layout()
    plt.show()

    return {
        "tested_settings": results,
        "best_n_estimators": best_n,
        "best_val_macro_f1": best_macro_f1
    }


start_time_tafm = now()
tuning_report = train_and_finetune_model(train_df, val_df)
end_time_tafm = now()


#############################################
# Documentation 4d & e & f
#############################################

# define these BEFORE using them
run1 = tuning_report["tested_settings"][0]
best_n = tuning_report["best_n_estimators"]
best_f1 = tuning_report["best_val_macro_f1"]

tafm_ass_uuid_writer = "d08c8893-379a-4af5-aaaf-e73dab042329"
tafm_comment = (
    f"Tuned n_estimators on the validation set. "
    f"Best n_estimators={best_n} (val_macro_f1={best_f1:.4f})."
)

training_run1 = "run_1"
model_run1 = "model_run1"
hp1_setting_run1 = "hp_setting_run1"
eval_validation_run1 = "metric_validation_run1"

train_model_activity = [
    # Activity
    ":train_and_finetune_model rdf:type prov:Activity .",
    ":train_and_finetune_model sc:isPartOf :modeling_phase .",
    f':train_and_finetune_model rdfs:comment "{tafm_comment}" .',
    f':train_and_finetune_model prov:startedAtTime "{start_time_tafm}"^^xsd:dateTime .',
    f':train_and_finetune_model prov:endedAtTime "{end_time_tafm}"^^xsd:dateTime .',
    f":train_and_finetune_model prov:qualifiedAssociation :{tafm_ass_uuid_writer} .",
    f":{tafm_ass_uuid_writer} prov:agent :{model_data_code_writer} .",
    f":{tafm_ass_uuid_writer} rdf:type prov:Association .",
    f":{tafm_ass_uuid_writer} prov:hadRole :{code_writer_role} .",

    # ONE documented run (run1)
    f":{hp1_setting_run1} rdf:type mls:HyperParameterSetting .",
    f":{hp1_setting_run1} mls:specifiedBy :hp_n_estimators .",
    f':{hp1_setting_run1} mls:hasValue "{int(run1["n_estimators"])}"^^xsd:integer .',
    f":{hp1_setting_run1} prov:wasGeneratedBy :train_and_finetune_model .",

    f":{training_run1} rdf:type mls:Run .",
    f":{training_run1} sc:isPartOf :train_and_finetune_model .",
    f":{training_run1} mls:realizes :random_forest_algorithm .",
    f':{training_run1} rdfs:label "Training Run 1 (n_estimators={int(run1["n_estimators"])})" .',
    f":{training_run1} mls:executes :random_forest_classifier_implementation .",
    f":{training_run1} mls:hasInput :training_set .",
    f":{training_run1} mls:hasInput :validation_set .",
    f":{training_run1} mls:hasInput :{hp1_setting_run1} .",
    f":{training_run1} mls:hasOutput :{model_run1} .",
    f":{training_run1} mls:hasOutput :{eval_validation_run1} .",

    f":{model_run1} rdf:type mls:Model .",
    f':{model_run1} rdfs:label "RandomForest model (Run 1)" .',
    f":{model_run1} prov:wasGeneratedBy :{training_run1} .",
    f":{model_run1} mlso:trainedOn :training_set .",
    f":{model_run1} mlso:hasAlgorithmType :random_forest_algorithm .",

    f":{eval_validation_run1} rdf:type mls:ModelEvaluation .",
    f":{eval_validation_run1} prov:wasGeneratedBy :{training_run1} .",
    f':{eval_validation_run1} mls:hasValue "{float(run1["val_macro_f1"])}"^^xsd:double .',
    f":{eval_validation_run1} mls:specifiedBy :macro_f1_measure .",
    f":{eval_validation_run1} prov:used :validation_set .",
    f':{eval_validation_run1} rdfs:comment "val_accuracy={float(run1["val_accuracy"])}" .',

    # 4f decision
    ":outcome_best_hp rdf:type prov:Entity .",
    f':outcome_best_hp rdfs:comment "Best n_estimators={best_n} based on validation Macro F1 ({best_f1:.4f})." .',
    ":outcome_best_hp prov:wasGeneratedBy :train_and_finetune_model .",
]

engine.insert(train_model_activity, prefixes=prefixes)


In [48]:
def retrain_model_full_data(training_set: pd.DataFrame, validation_set: pd.DataFrame, best_n_estimators: int):
    full_train = pd.concat([training_set, validation_set], ignore_index=True)

    X_full = full_train.drop(columns=["Class"])
    y_full = full_train["Class"]

    model = RandomForestClassifier(
        n_estimators=best_n_estimators,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_full, y_full)
    return model


best_n = tuning_report["best_n_estimators"]

start_time_retrain = now()
final_model_obj = retrain_model_full_data(train_df, val_df, best_n_estimators=best_n)
end_time_retrain = now()


#############################################
# Documentation 4g
#############################################

retrain_ass_uuid_writer = "d9402234-6663-48bf-a6f2-1efd223857db"  # Generate once

final_training_activity = ":retrain_final_model"
final_model = ":final_model_entity"

retrain_comment = (
    f"The final Random Forest model was retrained on the combined training and validation data "
    f"using the selected hyperparameter setting n_estimators={best_n} (random_state=42)."
)



retrain_documentation = [
    f"{final_training_activity} rdf:type prov:Activity .",
    f"{final_training_activity} sc:isPartOf :modeling_phase .",
    f'{final_training_activity} rdfs:comment "{retrain_comment}" .',
    f'{final_training_activity} prov:startedAtTime "{start_time_retrain}"^^xsd:dateTime .',
    f'{final_training_activity} prov:endedAtTime "{end_time_retrain}"^^xsd:dateTime .',

    f"{final_training_activity} prov:qualifiedAssociation :{retrain_ass_uuid_writer} .",
    f":{retrain_ass_uuid_writer} prov:agent :{model_data_code_writer} .",
    f":{retrain_ass_uuid_writer} rdf:type prov:Association .",
    f":{retrain_ass_uuid_writer} prov:hadRole :{code_executor_role} .",

    f"{final_training_activity} prov:used :training_set .",
    f"{final_training_activity} prov:used :validation_set .",

    f"{final_model} rdf:type mls:Model .",
    f'{final_model} rdfs:comment "Final Random Forest model trained on train+validation." .',
    f"{final_model} prov:wasGeneratedBy {final_training_activity} .",
    f"{final_model} mlso:trainedOn :training_set .",
    f"{final_model} mlso:trainedOn :validation_set .",
]
engine.insert(retrain_documentation, prefixes=prefixes)


## Evaluation

In [ ]:
## Each Activity that follows is part of the Evaluation Phase

evaluation_phase_executor = [
    ":evaluation_phase rdf:type prov:Activity .",
    ':evaluation_phase rdfs:label "Evaluation Phase" .',
]
engine.insert(evaluation_phase_executor, prefixes=prefixes)

In [ ]:
def evaluate_on_test_data(final_model: ClassifierMixin, test_set: pd.DataFrame) -> dict:
    # Identify one-hot columns and class label column
    onehot_cols = [col for col in test_set.columns if col.startswith("Class_")]
    # class_label_col = [col for col in test_set.columns if col not in onehot_cols][0]

    X_test = test_set.drop(onehot_cols, axis=1, errors="ignore")
    y_true = test_set[onehot_cols].to_numpy().argmax(axis=1)

    # Predict
    y_pred_proba = final_model.predict(X_test)
    if y_pred_proba.ndim > 1 and y_pred_proba.shape[1] == len(onehot_cols):
        y_pred = y_pred_proba.argmax(axis=1)
    else:
        y_pred = y_pred_proba

    # Macro metrics
    macro_accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")

    # Per-class (micro) precision and recall
    per_class_precision = precision_score(
        y_true, y_pred, average=None, labels=range(len(onehot_cols))
    )
    per_class_recall = recall_score(
        y_true, y_pred, average=None, labels=range(len(onehot_cols))
    )

    return {
        "macro_accuracy": macro_accuracy,
        "macro_f1": macro_f1,
        "per_class_precision": dict(zip(onehot_cols, per_class_precision)),
        "per_class_recall": dict(zip(onehot_cols, per_class_recall)),
    }


def compute_shap_feature_importance(final_model, test_set: pd.DataFrame) -> dict:
    # Identify feature columns (exclude one-hot and class label columns)
    onehot_cols = [col for col in test_set.columns if col.startswith("Class_")]
    class_label_col = [col for col in test_set.columns if col not in onehot_cols][0]
    X_test = test_set.drop(onehot_cols + [class_label_col], axis=1, errors="ignore")

    explainer = shap.Explainer(final_model, X_test)
    shap_values = explainer(X_test)
    # Compute mean absolute SHAP value for each feature (global importance)
    feature_importance = dict(
        zip(X_test.columns, np.abs(shap_values.values).mean(axis=0))
    )

    return feature_importance


def evaluation(
    final_model: ClassifierMixin, test_set: pd.DataFrame, train_set: pd.DataFrame
):
    eval_results = evaluate_on_test_data(final_model, test_set)
    feature_importance = compute_shap_feature_importance(final_model, test_set)

    # Baseline comparison:
    dummy = DummyClassifier(strategy="most_frequent")
    onehot_cols = [col for col in train_set.columns if col.startswith("Class_")]
    X_train = train_set.drop(onehot_cols, axis=1, errors="ignore")
    y_train = train_set[onehot_cols].to_numpy().argmax(axis=1)
    dummy.fit(X_train, y_train)
    dummy_eval_results = evaluate_on_test_data(dummy, test_set)

    return {
        "baseline": dummy_eval_results,
        "model": eval_results,
        "feature_importance": feature_importance,
    }

In [ ]:
eval_code_writer = student_b


start_time_eval = now()
# evaluation(model, test_df, train_df)
end_time_eval = now()

#############################################
# Documentation
#############################################

eval_ass_uuid = "47973081-8135-470e-96d4-8400182bcbb6"  # Generate once
final_model = ":final_model_entity"
test_set = ":test_set"

eval_comment = """
...
"""

evaluate_activity = [
    ":evaluate_final_model rdf:type prov:Activity .",
    ":evaluate_final_model sc:isPartOf :evaluation_phase .",
    ':evaluate_final_model rdfs:label "Final Model Evaluation on Test Set" .',
    f':evaluate_final_model rdfs:comment """{eval_comment}""" .',
    f':evaluate_final_model prov:startedAtTime "{start_time_eval}"^^xsd:dateTime .',
    f':evaluate_final_model prov:endedAtTime "{end_time_eval}"^^xsd:dateTime .',
    f":evaluate_final_model prov:qualifiedAssociation :{eval_ass_uuid} .",
    f":{eval_ass_uuid} prov:agent :{eval_code_writer} .",
    f":{eval_ass_uuid} rdf:type prov:Association .",
    f":{eval_ass_uuid} prov:hadRole :{code_writer_role} .",
    # Inputs
    f":evaluate_final_model prov:used {final_model} .",
    f":evaluate_final_model prov:used {test_set} .",
    # Reference to Data Mining Success Criteria from Phase 1
    ":evaluate_final_model prov:used :bu_data_mining_success_criteria .",
    # Document you final model performance
    # Hint: you evaluate bias in this way:
    ":bias_evaluation_result rdf:type mls:ModelEvaluation .",
    ":bias_evaluation_result prov:wasGeneratedBy :evaluate_final_model .",
    ':bias_evaluation_result rdfs:label "Bias Analysis" .',
    ':bias_evaluation_result rdfs:comment "..." .',
]
engine.insert(evaluate_activity, prefixes=prefixes)

## Deployment

In [ ]:
## Each Activity that follows is part of the Deployment Phase

deployment_phase_executor = [
    ":deployment_phase rdf:type prov:Activity .",
    ':deployment_phase rdfs:label "Deployment Phase" .',
]
engine.insert(deployment_phase_executor, prefixes=prefixes)

In [ ]:
#############################################
# Documentation
#############################################

comparison_and_recommendations_comment = """
...
"""

ethical_aspects_comment = """
...
"""

monitoring_plan_comment = """
...
"""

reproducibility_reflection_comment = """
...
"""

dep_ass_uuid_executor = "66979231-c456-47a0-8788-6b6b815e494f"  # Generate once
deployment_executor = [
    ":plan_deployment rdf:type prov:Activity .",
    ":plan_deployment sc:isPartOf :deployment_phase .",  # Connect to Parent Phase
    ':plan_deployment rdfs:label "Plan Deployment"@en .',
    f":plan_deployment prov:qualifiedAssociation :{dep_ass_uuid_executor} .",
    f":{dep_ass_uuid_executor} prov:agent :{executed_by} .",
    f":{dep_ass_uuid_executor} rdf:type prov:Association .",
    f":{dep_ass_uuid_executor} prov:hadRole :{code_executor_role} .",
]
engine.insert(deployment_executor, prefixes=prefixes)


deployment_data_executor = [
    # 6a
    ":dep_recommendations rdf:type prov:Entity .",
    ":dep_recommendations prov:wasGeneratedBy :plan_deployment .",
    ':dep_recommendations rdfs:label "6a Business Objectives Reflection and Deployment Recommendations" .',
    f':dep_recommendations rdfs:comment """{comparison_and_recommendations_comment}""" .',
    # 6b
    ":dep_ethical_risks rdf:type prov:Entity .",
    ":dep_ethical_risks prov:wasGeneratedBy :plan_deployment .",
    ':dep_ethical_risks rdfs:label "6b Ethical Aspects and Risks" .',
    f':dep_ethical_risks rdfs:comment """{ethical_aspects_comment}""" .',
    # 6c
    ":dep_monitoring_plan rdf:type prov:Entity .",
    ":dep_monitoring_plan prov:wasGeneratedBy :plan_deployment .",
    ':dep_monitoring_plan rdfs:label "6c Monitoring Plan" .',
    f':dep_monitoring_plan rdfs:comment """{monitoring_plan_comment}""" .',
    # 6d
    ":dep_reproducibility_reflection rdf:type prov:Entity .",
    ":dep_reproducibility_reflection prov:wasGeneratedBy :plan_deployment .",
    ':dep_reproducibility_reflection rdfs:label "6d Reproducibility Reflection" .',
    f':dep_reproducibility_reflection rdfs:comment """{reproducibility_reflection_comment}""" .',
]
engine.insert(deployment_data_executor, prefixes=prefixes)

# Generate Latex Report

The following cells give you an example of how to automatically create a Latex Report from your provenance documentation.

Feel free to use the example provided. If you use it, you should adapt and extend it with relevant sections/tables/plots/... 

In [ ]:
base_iri = f"https://starvers.ec.tuwien.ac.at/BI2025/{group_id}/"

In [ ]:
def latex_escape(text: str | None) -> str:
    if text is None:
        return ""
    text = str(text)
    text = text.replace("\\", r"\textbackslash{}")
    pairs = [
        ("&", r"\&"),
        ("%", r"\%"),
        ("$", r"\$"),
        ("#", r"\#"),
        ("_", r"\_"),
        ("{", r"\{"),
        ("}", r"\}"),
        ("~", r"\textasciitilde{}"),
        ("^", r"\textasciicircum{}"),
    ]
    for k, v in pairs:
        text = text.replace(k, v)
    return text


def clean_rdf(x) -> str:
    if hasattr(x, "toPython"):
        return str(x.toPython())
    if x is None:
        return ""
    s = str(x).strip()
    s = s.strip('"').strip("'")
    s = s.strip()
    if "^^" in s:
        s = s.split("^^")[0].strip('"')

    return s


def fmt_iso(ts: str) -> str:
    if not ts:
        return ""
    try:
        clean_ts = ts.split("^^")[0].strip('"')
        clean_ts = (
            clean_ts.replace("Z", "+00:00") if clean_ts.endswith("Z") else clean_ts
        )
        return datetime.datetime.fromisoformat(clean_ts).strftime("%Y-%m-%d %H:%M:%S")
    except:
        return latex_escape(str(ts))

In [ ]:
# This cell includes exemplary queries for different phases


### Author Block
author_query = f"""
{prefix_header}
PREFIX iao: <http://purl.obolibrary.org/obo/>

SELECT DISTINCT ?uri ?given ?family ?matr WHERE {{
  VALUES ?uri {{ :{student_a} :{student_b} }}
  
  ?uri a foaf:Person .
  ?uri foaf:givenName ?given .
  ?uri foaf:familyName ?family .
  ?uri iao:IAO_0000219 ?matr .
}}
"""

res_authors = engine.query(author_query)
author_block_latex = ""

if not res_authors.empty:  # type:ignore
    for _, row in res_authors.iterrows():  # type:ignore
        uri_str = str(row["uri"])
        given = latex_escape(clean_rdf(row["given"]))
        family = latex_escape(clean_rdf(row["family"]))
        matr = latex_escape(clean_rdf(row["matr"]))
        if student_a in uri_str:
            responsibility = "Student A"
        elif student_b in uri_str:
            responsibility = "Student B"
        else:
            responsibility = "Student"

        author_block_latex += rf"""
          \author{{{given} {family}}}
          \authornote{{{responsibility}, Matr.Nr.: {matr}}}
          \affiliation{{
            \institution{{TU Wien}}
            \country{{Austria}}
          }}
          """

### Business Understanding example
bu_query = f"""
{prefix_header}

SELECT ?ds_comment ?bo_comment ?bsc_comment ?dmg_comment ?dmsc_comment ?ai_risk_comment WHERE {{
  OPTIONAL {{ :bu_data_source_and_scenario rdfs:comment ?ds_comment . }}
  OPTIONAL {{ :bu_business_objectives rdfs:comment ?bo_comment . }}
  OPTIONAL {{ :bu_business_success_criteria rdfs:comment ?bsc_comment . }}
  OPTIONAL {{ :bu_data_mining_goals rdfs:comment ?dmg_comment . }}
  OPTIONAL {{ :bu_data_mining_success_criteria rdfs:comment ?dmsc_comment . }}
  OPTIONAL {{ :bu_ai_risk_aspects rdfs:comment ?ai_risk_comment . }}
}} LIMIT 1
"""
res_bu = engine.query(bu_query)
row_bu = res_bu.iloc[0] if not res_bu.empty else {}  # type:ignore
bu_data_source = latex_escape(clean_rdf(row_bu.get("ds_comment", "")))
bu_objectives = latex_escape(clean_rdf(row_bu.get("bo_comment", "")))
bu_success_criteria = latex_escape(clean_rdf(row_bu.get("bsc_comment", "")))
bu_dmg = latex_escape(clean_rdf(row_bu.get("dmg_comment", "")))
bu_dmsc = latex_escape(clean_rdf(row_bu.get("dmsc_comment", "")))
bu_ai_risks = latex_escape(clean_rdf(row_bu.get("ai_risk_comment", "")))


### Data Understanding examples
# Example Dataset Description
du_desc_query = f"""
{prefix_header}
SELECT ?desc WHERE {{ :raw_data sc:description ?desc . }} LIMIT 1
"""
res_du_desc = engine.query(du_desc_query)
row_du_desc = res_du_desc.iloc[0] if not res_du_desc.empty else {}  # type:ignore
du_description = latex_escape(clean_rdf(row_du_desc.get("desc", "")))

# Example Feature Columns Table
du_query = f"""
{prefix_header}

SELECT ?name (SAMPLE(?dtypeRaw) as ?dtype) (SAMPLE(?descRaw) as ?desc) WHERE {{
  :raw_data cr:recordSet ?rs .
  ?rs cr:field ?field .
  ?field sc:name ?name .
  ?field sc:description ?descRaw .
  ?field cr:dataType ?dtypeRaw .
}} 
GROUP BY ?name
ORDER BY ?name
"""
res_du = engine.query(du_query)
du_rows = []
if not res_du.empty:  # type:ignore
    for _, f in res_du.iterrows():  # type:ignore
        dtype_raw = clean_rdf(f.get("dtype", ""))
        if "#" in dtype_raw:
            dtype = dtype_raw.split("#")[-1]
        elif "/" in dtype_raw:
            dtype = dtype_raw.split("/")[-1]
        else:
            dtype = dtype_raw

        desc = clean_rdf(f.get("desc", ""))
        row_str = f"{latex_escape(clean_rdf(f['name']))} & {latex_escape(dtype)} & {latex_escape(desc)} \\\\"
        du_rows.append(row_str)
du_table_rows = "\n    ".join(du_rows)

# Check outliers and outlier report
du_outlier_query = f"""
{prefix_header}
SELECT ?outlier_comment WHERE {{
  :check_outliers rdfs:comment ?outlier_comment .
}} LIMIT 1
"""
res_du_outlier = engine.query(du_outlier_query)
res_du_outlier = res_du_outlier.iloc[0] if not res_du_outlier.empty else {}  # type:ignore
du_outlier_analysis = latex_escape(clean_rdf(res_du_outlier.get("outlier_comment", "")))

du_outlier_report_query = f"""
{prefix_header}
SELECT ?outlier_comment WHERE {{
  :inspect_outlier_report rdfs:comment ?outlier_comment .
}} LIMIT 1
"""
res_du_outlier = engine.query(du_outlier_report_query)
res_du_outlier = res_du_outlier.iloc[0] if not res_du_outlier.empty else {}  # type:ignore
du_outlier_report = latex_escape(clean_rdf(res_du_outlier.get("outlier_comment", "")))

# Statistical properties
du_stats_query = f"""
{prefix_header}
SELECT ?stat_prop WHERE {{
  :stat_properties rdfs:comment ?stat_prop .
}} LIMIT 1
"""
res_du_stats = engine.query(du_stats_query)
res_du_stats = res_du_stats.iloc[0] if not res_du_stats.empty else {}  # type:ignore
du_statistical_properties = latex_escape(clean_rdf(res_du_stats.get("stat_prop", "")))

# Data quality
du_dq_query = f"""
{prefix_header}
SELECT ?dq_comment WHERE {{
  :data_quality rdfs:comment ?dq_comment .
}} LIMIT 1
"""
res_du_dq = engine.query(du_dq_query)
res_du_dq = res_du_dq.iloc[0] if not res_du_dq.empty else {}  # type:ignore
du_data_quality = latex_escape(clean_rdf(res_du_dq.get("dq_comment", "")))

# Visual exploration
du_vis_query = f"""
{prefix_header}
SELECT ?vis_comment WHERE {{
  :visual_exploration rdfs:comment ?vis_comment .
}} LIMIT 1
"""
res_du_vis = engine.query(du_vis_query)
res_du_vis = res_du_vis.iloc[0] if not res_du_vis.empty else {}  # type:ignore
du_data_visual_exploration = latex_escape(clean_rdf(res_du_vis.get("vis_comment", "")))

# Ethical and AI risk comments + data preparation actions
du_ea_query = f"""
{prefix_header}
SELECT ?ea_comment WHERE {{
  :ethical_aspects rdfs:comment ?ea_comment .
}} LIMIT 1
"""
res_du_ea = engine.query(du_ea_query)
res_du_ea = res_du_ea.iloc[0] if not res_du_ea.empty else {}  # type:ignore
du_ai_ethics_aspects = latex_escape(clean_rdf(res_du_ea.get("ea_comment", "")))

### Modeling example
# Hyperparameters
hp_query = f"""
{prefix_header}

SELECT ?hpName (SAMPLE(?hpValRaw) as ?hpVal) (MAX(?hpDescRaw) as ?hpDesc) WHERE {{
  ?run sc:isPartOf :train_and_finetune_model .
  ?run mls:hasInput ?setting .
  ?setting a mls:HyperParameterSetting .
  ?setting mls:hasValue ?hpValRaw .
  ?setting mls:specifiedBy ?hpDef .
  ?hpDef rdfs:label ?hpName .
  OPTIONAL {{ ?hpDef rdfs:comment ?hpDescRaw . }}
}} 
GROUP BY ?hpName
ORDER BY ?hpName
"""
res_hp = engine.query(hp_query)
hp_rows = []
if not res_hp.empty:  # type:ignore
    for _, row in res_hp.iterrows():  # type:ignore
        name = latex_escape(clean_rdf(row["hpName"]))
        val = latex_escape(clean_rdf(row["hpVal"]))
        desc = latex_escape(clean_rdf(row.get("hpDesc", "")))
        hp_rows.append(rf"{name} & {desc} & {val} \\")

hp_table_rows = "\n    ".join(hp_rows)

# Run Info
run_query = f"""
{prefix_header}

SELECT ?algoLabel ?start ?end ?metricLabel ?metricVal WHERE {{
  OPTIONAL {{ :train_and_finetune_model prov:startedAtTime ?start ; prov:endedAtTime ?end . }}
  OPTIONAL {{
      ?run sc:isPartOf :train_and_finetune_model .
      ?run mls:realizes ?algo .
      ?algo rdfs:label ?algoLabel .
  }}
  OPTIONAL {{
    ?run sc:isPartOf :train_and_finetune_model .
    ?run mls:hasOutput ?eval .
    ?eval a mls:ModelEvaluation ; mls:hasValue ?metricVal .
    OPTIONAL {{ ?eval mls:specifiedBy ?m . ?m rdfs:label ?metricLabel . }}
  }}
}} LIMIT 1
"""
res_run = engine.query(run_query)
row_run = res_run.iloc[0] if not res_run.empty else {}  # type:ignore
mod_algo = latex_escape(clean_rdf(row_run.get("algoLabel", "")))
mod_start = latex_escape(fmt_iso(clean_rdf(row_run.get("start"))))
mod_end = latex_escape(fmt_iso(clean_rdf(row_run.get("end"))))
mod_m_lbl = latex_escape(clean_rdf(row_run.get("metricLabel", "")))
raw_val = clean_rdf(row_run.get("metricVal", ""))
mod_m_val = f"{float(raw_val):.4f}" if raw_val else ""

print("Data extraction done.")

The following includes the Latex report itself. It fills in the query-results from the cell before. The ACM Template is already filled. 
Make sure that you update Student A and B accordingly.

In [ ]:
latex_content = rf"""\documentclass[sigconf]{{acmart}}

\AtBeginDocument{{ \providecommand\BibTeX{{ Bib\TeX }} }}
\setcopyright{{acmlicensed}}
\copyrightyear{{2025}}
\acmYear{{2025}}
\acmDOI{{XXXXXXX.XXXXXXX}}

\acmConference[BI 2025]{{Business Intelligence}}{{-}}{{-}}

\begin{{document}}

\title{{BI2025 Experiment Report - Group {group_id}}}
%% ---Authors: Dynamically added ---
{author_block_latex}

\begin{{abstract}}
  This report documents the machine learning experiment for Group {group_id}, following the CRISP-DM process model.
\end{{abstract}}

\ccsdesc[500]{{Computing methodologies~Machine learning}}
\keywords{{CRISP-DM, Provenance, Knowledge Graph, Machine Learning}}

\maketitle

%% --- 1. Business Understanding ---
\section{{Business Understanding}}

\subsection{{Data Source and Scenario}}
{bu_data_source}

\subsection{{Business Objectives}}
{bu_objectives}

\subsection{{Business Success Criteria}}
{bu_success_criteria}

\subsection{{Data Mining Goals}}
{bu_dmg}

\subsection{{Data Mining Success Criteria}}
{bu_dmsc}

\subsection{{AI Risk Aspects}}
{bu_ai_risks}

%% --- 2. Data Understanding ---
\section{{Data Understanding}}
\textbf{{Dataset Description:}} {du_description}

The following features were identified in the dataset:

\begin{{table}}[h]
  \caption{{Raw Data Features}}
  \label{{tab:features}}
  \begin{{tabular}}{{lp{{0.2\linewidth}}p{{0.4\linewidth}}}}
    \toprule
    \textbf{{Feature Name}} & \textbf{{Data Type}} & \textbf{{Description}} \\
    \midrule
    {du_table_rows}
    \bottomrule
  \end{{tabular}}
\end{{table}}

\subsection{{Outlier Analysis}}
{du_outlier_analysis}

{du_outlier_report}

\subsection{{Statistical Properties}}
{du_statistical_properties}

\subsection{{Data Quality Assessment}}
{du_data_quality}

\subsection{{Visual Data Exploration}}
{du_data_visual_exploration}

\subsection{{Ethical and AI Risk Aspects}}
{du_ai_ethics_aspects}

%% --- 3. Data Preparation ---
\section{{Data Preparation}}
\subsection{{Data Cleaning}}
Describe your Data preparation steps here and include respective graph data.


%% --- 4. Modeling ---
\section{{Modeling}}

\subsection{{Hyperparameter Configuration}}
The model was trained using the following hyperparameter settings:

\begin{{table}}[h]
  \caption{{Hyperparameter Settings}}
  \label{{tab:hyperparams}}
  \begin{{tabular}}{{lp{{0.4\linewidth}}l}}
    \toprule
    \textbf{{Parameter}} & \textbf{{Description}} & \textbf{{Value}} \\
    \midrule
    {hp_table_rows}
    \bottomrule
  \end{{tabular}}
\end{{table}}

\subsection{{Training Run}}
A training run was executed with the following characteristics:
\begin{{itemize}}
    \item \textbf{{Algorithm:}} {mod_algo}
    \item \textbf{{Start Time:}} {mod_start}
    \item \textbf{{End Time:}} {mod_end}
    \item \textbf{{Result:}} {mod_m_lbl} = {mod_m_val}
\end{{itemize}}

%% --- 5. Evaluation ---
\section{{Evaluation}}

%% --- 6. Deployment ---
\section{{Deployment}}

\section{{Conclusion}}

\end{{document}}
"""

In [ ]:
# This cell stores the Latex report to the data/report directory

out_dir = os.path.join("data", "report")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "experiment_report.tex")

with open(out_path, "w", encoding="utf-8") as f:
    f.write(latex_content)

print(f"Report written to: {out_path}")